<a href="https://colab.research.google.com/github/tassiana-bastos/my_portfolio/blob/%F0%9F%9A%80-Big-Data-Apache-Spark/Apache_Spark_Rick_and_Morty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#The libraries are being downloaded here.
!pip install Pyspark
!pip install GraphFrames
!pip install findspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
#Here we are calling the modules that we will need to use in the code.
from pyspark.sql import SparkSession
from pyspark.sql import Row
from graphframes import GraphFrame

In [ ]:
#Below, a Spark session is being created, called “Rick and Morty Multiverse,” where we can store the codes.
spark = SparkSession.builder \
    .appName("Multiverso de Rick and Morty") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.4-spark3.5-s_2.12") \
    .getOrCreate()


In [ ]:
#The variable “vertices” stores the lines relating to the characters and distributes them into three columns: “id,” “nome,” and “tipo.”
#The variable "arestas" stores the relationships between characters, defining the relationship between one and another through the fields
#“source” as “src,” ‘destination’ as “dst,” and “relacao”.
vertices = spark.createDataFrame([
    ("r1", "Rick C-137", "Rick"),
    ("m1", "Morty C-137", "Morty"),
    ("r2", "Rick D-99", "Rick"),
    ("m2", "Morty D-99", "Morty"),
    ("s1", "Summer C-137", "Summer"),
    ("b1", "Birdperson", "Aliado"),
    ("t1", "Tammy", "Inimigo"),
    ("p1", "Phoenixperson", "Inimigo"),
], ["id", "nome", "tipo"])

arestas = spark.createDataFrame([
    ("r1", "m1", "protege"),
    ("r1", "s1", "protege"),
    ("m1", "s1", "amigo"),
    ("r2", "m2", "protege"),
    ("b1", "r1", "amigo"),
    ("t1", "b1", "traiu"),
    ("p1", "r1", "inimigo"),
    ("r1", "r2", "conhece"),
    ("m1", "m2", "conhece"),
    ("r2", "t1", "suspeita")
], ["src", "dst", "relacao"])



In [ ]:
#The above variables are being called through the GraphFrame function.
gf = GraphFrame(vertices, arestas)

/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


In [ ]:
#Below, the show() function prints the graphs related to the vertices and edges tables.
gf.vertices.show()
gf.edges.show()


+---+-------------+-------+
| id|         nome|   tipo|
+---+-------------+-------+
| r1|   Rick C-137|   Rick|
| m1|  Morty C-137|  Morty|
| r2|    Rick D-99|   Rick|
| m2|   Morty D-99|  Morty|
| s1| Summer C-137| Summer|
| b1|   Birdperson| Aliado|
| t1|        Tammy|Inimigo|
| p1|Phoenixperson|Inimigo|
+---+-------------+-------+

+---+---+--------+
|src|dst| relacao|
+---+---+--------+
| r1| m1| protege|
| r1| s1| protege|
| m1| s1|   amigo|
| r2| m2| protege|
| b1| r1|   amigo|
| t1| b1|   traiu|
| p1| r1| inimigo|
| r1| r2| conhece|
| m1| m2| conhece|
| r2| t1|suspeita|
+---+---+--------+



In [ ]:
#Here, the inDegrees method is being used, which calculates how many edges (connections) point to each vertice.
gf.inDegrees.show()


/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+--------+
| id|inDegree|
+---+--------+
| r1|       2|
| m1|       1|
| s1|       2|
| m2|       2|
| r2|       1|
| t1|       1|
| b1|       1|
+---+--------+



In [ ]:
#The outDegrees method, on the other hand, calculates how many edges (connections) leave each vertice.
gf.outDegrees.show()


+---+---------+
| id|outDegree|
+---+---------+
| r1|        3|
| r2|        2|
| b1|        1|
| m1|        2|
| t1|        1|
| p1|        1|
+---+---------+



In [ ]:
#Here, the Pyspark.sql join function is being applied to join the outDegrees connections via the “id” while bringing the
#information from the “vertices” DataFrame, including name and type.
graus = gf.outDegrees.join(gf.vertices, "id")

#Using orderBy and ascending=False, the graph will be organized from the vertice with the highest number of outDegree connections to the vertice with the lowest
#number of outDegree connections. Only the first row will be shown, i.e., the vertice with the highest number of connections, due to show(1).
graus.orderBy("outDegree", ascending=False).select("nome", "outDegree").show(1)


/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+----------+---------+
|      nome|outDegree|
+----------+---------+
|Rick C-137|        3|
+----------+---------+
only showing top 1 row



In [ ]:
#Using the find method and its syntax, the relationship below is found.
padrao = gf.find("(r)-[e1]->(m); (m)-[e2]->(s)").filter(
"r.tipo = 'Rick' and m.tipo= 'Morty' and s.tipo = 'Summer' and e1.relacao = 'protege' and e2.relacao = 'amigo'")
padrao.select("r.nome", "m.nome", "s.nome").show()

+----------+-----------+------------+
|      nome|       nome|        nome|
+----------+-----------+------------+
|Rick C-137|Morty C-137|Summer C-137|
+----------+-----------+------------+



In [ ]:
#The PageRank algorithm is being used to find the three vertices with the highest degree of influence.
#The following SQL functions are being used to organize the graph.
pagerank = gf.pageRank(resetProbability=0.15, maxIter=10)
pagerank.vertices.select("id", "nome", "pagerank").orderBy("pagerank",
ascending=False).show(3)


/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


+---+------------+------------------+
| id|        nome|          pagerank|
+---+------------+------------------+
| r1|  Rick C-137| 1.648089014465783|
| s1|Summer C-137| 1.240534603943088|
| m2|  Morty D-99|1.1379657092363857|
+---+------------+------------------+
only showing top 3 rows



In [ ]:
#Through breadth-first search (BFS), a distance of no more than 3 steps between Rick C-137 and Phoenixperson is being sought.
#Since none was found in this case, an empty graph is displayed.

caminho = gf.bfs(fromExpr="nome = 'Rick C-137'", toExpr="nome = 'Phoenixperson'",
maxPathLength=3)
caminho.show(truncate=False)


+---+----+----+
|id |nome|tipo|
+---+----+----+
+---+----+----+



In [ ]:
#First, a directory is created to store the checkpoints (a mandatory requirement for using the connectedComponents method).
#In a second step, the code searches for and returns how many components belong to this graph and which characters are associated with this component.

spark.sparkContext.setCheckpointDir("/tmp/checkpoints")
components = gf.connectedComponents()
components.select("id", "nome", "component").orderBy("component").show()
components.select("component").distinct().count()


+---+-------------+------------+
| id|         nome|   component|
+---+-------------+------------+
| r1|   Rick C-137|403726925824|
| s1| Summer C-137|403726925824|
| m1|  Morty C-137|403726925824|
| b1|   Birdperson|403726925824|
| r2|    Rick D-99|403726925824|
| t1|        Tammy|403726925824|
| m2|   Morty D-99|403726925824|
| p1|Phoenixperson|403726925824|
+---+-------------+------------+



1

In [ ]:
#Using the find algorithm, a cycle of three vertices connected to each other, i.e., A -> B -> c -> A, is being searched for.
#Since there is no such case, an empty graph is shown.
triangulos = gf.find("(a)-[e1]->(b); (b)-[e2]->(c); (c)-[e3]->(a)")
triangulos.show()

+---+---+---+---+---+---+
|  a| e1|  b| e2|  c| e3|
+---+---+---+---+---+---+
+---+---+---+---+---+---+



In [ ]:
#Loading vertices and edges into DataFrames.
vertices1 = spark.createDataFrame([
("r1", "Rick C-137"),
("r2", "Rick D-99"),
("r3", "Rick X-25"),
("m1", "Morty C-137"),
("m2", "Morty D-99"),
("m3", "Morty X-25"),
("p1", "Presidente Morty")
], ["id", "nome"])

arestas1 = spark.createDataFrame([
("r1", "m1", "protege"),
("r2", "m2", "protege"),
("r3", "m3", "protege"),
("m1", "p1", "desconfia"),
("m2", "p1", "desconfia"),
("m3", "p1", "desconfia"),
("p1", "r1", "controla"),
("p1", "r2", "controla")
], ["src", "dst", "relacao"])

gf2 = GraphFrame(vertices1, arestas1)



In [ ]:
#Calculating the shortest path from all vertices to “p1.”
caminhos = gf2.shortestPaths(landmarks=["p1"])
caminhos.select("id", "distances").show(truncate=False)


+---+---------+
|id |distances|
+---+---------+
|r1 |{p1 -> 2}|
|m3 |{p1 -> 1}|
|r2 |{p1 -> 2}|
|m1 |{p1 -> 1}|
|p1 |{p1 -> 0}|
|r3 |{p1 -> 2}|
|m2 |{p1 -> 1}|
+---+---------+



In [ ]:
from pyspark.sql.functions import col

#Filter the Rickys and see their distances.
caminhos.select("id", "distances").filter(col("id").startswith("r")).show(truncate=False)

#Filter the Rickys and see their distances to the President using distances.
ricks_distancias = caminhos.filter(col("id").startswith("r")).withColumn("distancia_p1", col("distances").getItem("p1"))


+---+---------+
|id |distances|
+---+---------+
|r1 |{p1 -> 2}|
|r2 |{p1 -> 2}|
|r3 |{p1 -> 2}|
+---+---------+



In [ ]:
#Remove the Ricks that do not have a path to the President.
ricks_com_caminho = ricks_distancias.filter(col("distancia_p1").isNotNull())

#Shows Rick with the shortest path to the President.
rick_mais_vulneravel = ricks_com_caminho.orderBy(col("distancia_p1").asc()).limit(1)

rick_mais_vulneravel.select("id", "nome", "distancia_p1").show()


+---+----------+------------+
| id|      nome|distancia_p1|
+---+----------+------------+
| r1|Rick C-137|           2|
+---+----------+------------+



In [ ]:
#The data is being loaded into the vertices3 and arestas3 DataFrames.

vertices3 = spark.createDataFrame([
("r1", "Rick C-137"),
("e1", "Evil Rick"),
("m1", "Morty C-137"),
("m4", "Evil Morty"),
("t1", "Tammy")
], ["id", "nome"])


arestas3 = spark.createDataFrame([
("r1", "e1", "suspeita"),
("e1", "m4", "controla"),
("m4", "r1", "manipula"),
("m1", "m4", "inveja"),
("t1", "e1", "aliado"),
("t1", "m4", "espionagem")
], ["src", "dst", "relacao"])


#creating the graph
gf3 = GraphFrame(vertices3, arestas3)

#The stronglyConnectedComponents algorithm is being called to perform a maximum iteration of 10x.
scc = gf3.stronglyConnectedComponents(maxIter=10)

#The components are being displayed using the show() function.
scc.select("id", "nome", "component").orderBy("component").show()


/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:168: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+---+-----------+-------------+
| id|       nome|    component|
+---+-----------+-------------+
| r1| Rick C-137| 403726925824|
| e1|  Evil Rick| 403726925824|
| m4| Evil Morty| 403726925824|
| t1|      Tammy| 807453851648|
| m1|Morty C-137|1305670057984|
+---+-----------+-------------+

